In [1]:
import pandas as pd
import numpy as np
import requests
import io
import yfinance as yf
import logging

**Data quality check and cleaning**

*For prices*

In [2]:
def download_prices(ticker_list, start_date, end_date):
    stocks = []
    stocks_not_downloaded = []
    for t in ticker_list:
        try:
            df = yf.download(t, start=start_date, end=end_date, progress=False, auto_adjust=False)
        except Exception as e:
            stocks_not_downloaded.append((t, e))
            continue
        if df.empty:
            stocks_not_downloaded.append((t, "no info"))
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df['Ticker'] = t
        stocks.append(df)
    df = pd.concat(stocks)
    return df, stocks_not_downloaded

In [3]:
df_tickers = pd.read_csv("../tickers.csv")
tickers_list = df_tickers['Ticker'].tolist()
df_prices, list_not_downloaded = download_prices(tickers_list, '2016-01-01', '2026-11-16')
df_prices = df_prices.reset_index()
df_prices.to_parquet('../data/prices.parquet')
list_not_downloaded

[]

In [4]:
df_prices.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Ticker
0,2017-06-23,1.698750,2.30,2.43,2.30,2.38,1504121,ADMIE.AT
1,2017-06-26,1.654435,2.24,2.34,2.22,2.30,957607,ADMIE.AT
2,2017-06-27,1.639663,2.22,2.25,2.17,2.24,979758,ADMIE.AT
3,2017-06-28,1.647049,2.23,2.24,2.17,2.22,1068257,ADMIE.AT
4,2017-06-29,1.669207,2.26,2.27,2.23,2.24,316102,ADMIE.AT


Check for duplicates

In [5]:
duplicates = df_prices[df_prices.duplicated(subset=["Ticker", "Date"], keep=False)]
len(duplicates)

0

Check NaN

In [6]:
df_prices.isna().sum()

Price
Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
Ticker       0
dtype: int64

Check Daily return

In [7]:
df_prices = df_prices.sort_values(by=['Ticker', 'Date'])
df_daily_turnover = df_prices.groupby('Ticker')['Adj Close'].pct_change()

s = df_daily_turnover[abs(df_daily_turnover)>0.4]
s

70005   -0.409922
Name: Adj Close, dtype: float64

In [8]:
df_prices.loc[70005]

Price
Date         2017-06-19 00:00:00
Adj Close               2.093913
Close                       2.26
High                        2.37
Low                         1.75
Open                        1.78
Volume                   4589602
Ticker                    PPC.AT
Name: 70005, dtype: object

In [9]:
df_prices[(df_prices['Ticker'] == "PPC.AT") & ((df_prices['Date'] >= '2017-06-14') & (df_prices['Date'] <= '2017-06-22'))]

Price,Date,Adj Close,Close,High,Low,Open,Volume,Ticker
70002,2017-06-14,3.548533,3.83,3.94,3.75,3.80,898659,PPC.AT
70003,2017-06-15,3.548533,3.83,3.83,3.83,3.83,0,PPC.AT
70004,2017-06-16,3.548533,3.83,3.83,3.83,3.83,0,PPC.AT
70005,2017-06-19,2.093913,2.26,2.37,1.75,1.78,4589602,PPC.AT
70006,2017-06-20,2.093913,2.26,2.33,2.20,2.31,1469332,PPC.AT
70007,2017-06-21,2.140238,2.31,2.35,2.26,2.26,766316,PPC.AT
70008,2017-06-22,2.168033,2.34,2.54,2.31,2.31,1005085,PPC.AT


Check the last date for information

In [10]:
df_last_date = df_prices.groupby('Ticker', as_index=False).agg(
    last_date=('Date', 'max')
)
df_last_date[df_last_date['last_date'] != '2026-11-13']    

,Ticker,last_date


One daily return beyond ±40% (TICKER PPC.AT, June 2017): both Close and Adj Close dropped equally after two zero-volume days, so it is a real price move, not a data error. Kept. 

*Checking Athens general composite index GD.AT*

In [11]:
tickers_list = ['GD.AT']
df_index, list_not_downloaded = download_prices(tickers_list, '2016-01-01', '2026-11-16')
df_index = df_index.reset_index()
df_index.to_parquet('../data/index.parquet')
list_not_downloaded

[]

Check for duplicates

In [12]:
duplicates = df_index[df_index.duplicated(subset=["Ticker", "Date"], keep=False)]
len(duplicates)

0

Check NaN

In [13]:
df_index.isna().sum()

Price
Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
Ticker       0
dtype: int64

Checking for full data (for other Tickers we checked on 01 notebook)

In [14]:
df_stat = df_index.groupby('Ticker', as_index=False).agg(
    first_date=('Date', 'min'),
    last_date=('Date', 'max'),
    count_dates = ('Date', 'count'),
    nill_dates = ('Volume', lambda x: (x == 0).sum()
                 )
)
df_stat

,Ticker,first_date,last_date,count_dates,nill_dates
0,GD.AT,2016-01-04,2026-11-13,2701,4


Download long-term interest rate Greece and Germany and deposit facility

In [15]:
url_de = 'https://data-api.ecb.europa.eu/service/data/IRS/M.DE.L.L40.CI.0000.EUR.N.Z?format=csvdata&startPeriod=2016-01-01'
url_gr = 'https://data-api.ecb.europa.eu/service/data/IRS/M.GR.L.L40.CI.0000.EUR.N.Z?format=csvdata&startPeriod=2016-01-01'
url_df = 'https://data-api.ecb.europa.eu/service/data/FM/D.U2.EUR.4F.KR.DFR.LEV?format=csvdata&startPeriod=2016-01-01'

In [16]:
def read_csv_data(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    print(resp.status_code)
    df = pd.read_csv(io.BytesIO(resp.content))
    return df

Germany

In [17]:
df_de = read_csv_data(url_de)
len(df_de)

200


130

Greece

In [18]:
df_gr = read_csv_data(url_gr)
len(df_gr)

200


130

In [19]:
df_df = read_csv_data(url_df)
len(df_df)

200


3976

Keeping only columns we need and convert types

In [20]:
df_gr = df_gr[['TIME_PERIOD', 'OBS_VALUE']].rename(columns={'TIME_PERIOD': 'date', 'OBS_VALUE': 'gr_10y'})
df_de = df_de[['TIME_PERIOD', 'OBS_VALUE']].rename(columns={'TIME_PERIOD': 'date', 'OBS_VALUE': 'de_10y'})
df_df = df_df[['TIME_PERIOD', 'OBS_VALUE']].rename(columns={'TIME_PERIOD': 'date', 'OBS_VALUE': 'deposit_rate'})

In [21]:
df_gr['date'] = pd.to_datetime(df_gr['date'])
df_de['date'] = pd.to_datetime(df_de['date'])
df_df['date'] = pd.to_datetime(df_df['date'])

In [22]:
df_bonds = pd.merge(df_gr, df_de, on='date', how='inner')
df_bonds = df_bonds.sort_values('date').reset_index(drop=True)
df_bonds.to_parquet('../data/bonds_10y.parquet', index=False)
df_df.to_parquet('../data/ecb_deposit_rate.parquet', index=False)